<a href="https://colab.research.google.com/github/muneer-ahmad10/End_Module_exam_prep/blob/main/weak_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The RAG Pipeline**

***one continuous story: raw PDF → chunks → embeddings → vector index → retriever → reader/generator → orchestration → full app.***

# **Pretrained Transformers for Embedding Generation & Fine-tuning**

### **Word2Vec-style embeddings**

one fixed vector per word, regardless of context ("bank" always got the same vector, whether river bank or money bank). Transformer-based embeddings (from BERT-style encoders) are contextual — the same word gets a different vector depending on the sentence it's in, because self-attention lets each token's representation absorb information from surrounding words.

**Sentence embeddings:** to represent a whole sentence/document (not just one word) as a single vector, you typically pool the token-level outputs — commonly by taking the [CLS] token's representation (Day 10-13 callback), or averaging all token embeddings (mean pooling). This single vector is what later gets stored and searched in Day 26's FAISS index

**Fine-tuning embeddings:** pretrained embedding models can be fine-tuned on domain-specific data (e.g. legal or medical text) so that "similar meaning" is defined in a way that matches your specific domain, rather than generic web text.

## **PDF Processing and Chunk-Level QA**

Before any embedding/search happens, you need raw text out of a PDF:

* Extract text from the PDF (handling multi-column layouts, tables, headers/footers as noise to potentially strip)

* Split ("chunk") the text into smaller pieces — because a full PDF is far too long to fit into a model's context window at once, and because retrieval works better on focused, topically-coherent pieces

**Chunk-Level QA:** given a single chunk of text and a question, extract the answer from that chunk specifically — this is the simplest form of the QA task, before you introduce the complexity of "which chunk even has the answer?"

## **Semantic-Aware Chunking and Semantic Embedding**

### **The problem with naive chunking**

The simplest chunking approach just splits text every N characters/words (e.g. every 500 words), regardless of where sentences or ideas actually end. This can slice a sentence or idea in half mid-thought — a chunk might end with "...the treatment showed significant" and the next chunk starts with "improvement in patients who...", breaking the semantic unit apart right at the boundary.

### **Semantic-aware chunking — the fix**

Instead of splitting by a fixed character/word count blindly, semantic chunking tries to split at natural topic/meaning boundaries — e.g. splitting by paragraph, by detecting where sentence embeddings shift significantly in meaning (indicating a topic change), or using sentence boundaries as safer split points rather than cutting mid-sentence.

**Common practical strategies:**

* **Recursive/hierarchical splitting**: try to split by paragraph first; if a paragraph is still too long, fall back to splitting by sentence; if a sentence is still too long, fall back to splitting by fixed size — always preferring the most "natural" boundary available

* **Overlap between chunks**: even with good boundaries, adding a small overlap (e.g. 50 words) between consecutive chunks helps preserve context that might otherwise be lost right at a boundary

* **Embedding-based splitting**: compute embeddings for consecutive sentences, and split where the similarity between adjacent sentence embeddings drops sharply — signaling a topic shift

**Why this matters for retrieval quality (the actual exam-relevant "why")**: if a chunk is split awkwardly mid-idea, its embedding ends up representing a confused, incomplete thought — making it less likely to be retrieved correctly when a relevant question comes in, even though the answer might technically be present in the (badly split) text.

### **Semantic Embedding**

This section reinforces the same core embedding concept, now explicitly framed for retrieval: each chunk gets converted into a dense vector using a pretrained (possibly fine-tuned) sentence/document embedding model — and the entire RAG system's retrieval quality depends on how good these embeddings are at placing semantically similar chunks close together in vector space.

## **Semantic Search with Cosine Similarity**

### **Why cosine similarity, specifically**

Once you have embeddings, you need a way to measure "how similar is this chunk to the question?" Cosine similarity measures the angle between two vectors, not their magnitude:

***cosine_similarity(A, B) = (A · B) / (‖A‖ × ‖B‖)***

* Result ranges from -1 to 1 (in practice, embedding similarity is usually 0 to 1, since embeddings from these models tend not to point in opposite directions)
* 1 = vectors point in the exact same direction (maximally similar)
* 0 = vectors are orthogonal (unrelated)

**Why cosine, not Euclidean distance** (a very common exam question): cosine similarity ignores vector magnitude and only cares about direction. This matters because embedding vector length can vary based on factors unrelated to meaning (e.g. text length, how the model happened to scale that particular output) — two chunks could express very similar meaning but have different magnitudes, and cosine similarity correctly still recognizes them as similar since it only looks at the angle between them. Euclidean distance would be thrown off by magnitude differences even when direction (meaning) is nearly identical.

## **The semantic search process, end to end**

1.   Embed the user's question using the same embedding model used for the chunks
2.   Compute cosine similarity between the question's embedding and every stored chunk's embedding
3.  Rank chunks by similarity score, return the top-K most similar ones
4.  Those top-K chunks become the "retrieved context" fed into the reader/generator in the RAG pipeline




## **Vector Search and FAISS Fundamentals**

### **What FAISS actually does**

FAISS is a library for efficient similarity search over large collections of vectors. Instead of comparing your query against every single stored vector one by one (brute force — call this "flat" search), FAISS builds an index structure that lets you find approximate nearest neighbors much faster, at the cost of a small accuracy trade-off.

### **Key FAISS concepts**

### **Flat index (IndexFlatL2 / IndexFlatIP):**

* This is literally brute-force — compares the query against every vector
* Exact results (no approximation), but slow at scale — this is your Day 26 baseline, just formalized as a named FAISS index type
* L2 = Euclidean distance, IP = Inner Product (related to cosine similarity when vectors are normalized)

## **Approximate Nearest Neighbor (ANN) indexes — the actual speedup:**

* **IVF (Inverted File Index):** clusters the vector space into groups (via k-means-like clustering) ahead of time. At search time, only compares the query against vectors in the nearest few clusters, not the whole dataset — massive speedup, small accuracy trade-off
* **HNSW (Hierarchical Navigable Small World):** builds a graph structure connecting nearby vectors, allowing fast traversal to find approximate nearest neighbors without visiting every vector — another common ANN approach

**The core trade-off to remember for exams**: Flat/exact search = 100% accurate but slow (scales linearly with dataset size). ANN methods (IVF, HNSW) = much faster, but return "approximately" the nearest neighbors — occasionally missing the true single best match in exchange for massive speed gains. This speed/accuracy trade-off is the single most tested FAISS concept.

### **Normalizing vectors — a practical detail worth knowing**

If you normalize all your vectors to unit length before indexing (making every vector length = 1), then Inner Product search becomes mathematically equivalent to cosine similarity search. This is why FAISS pipelines often normalize embeddings first, then use IndexFlatIP (or an ANN variant) — it lets you get cosine-similarity-style ranking while using FAISS's faster inner-product-optimized operations.

## **Building a FAISS-Based PDF Retriever**

### **The full pipeline, end to end**

PDF → Extract text → Chunk (semantic-aware) → Embed each chunk →
Build FAISS index → [Query time] Embed question → Search FAISS index → Return top-K chunks

### **Step-by-step build**

### **1. Indexing phase**

In [ ]:
# import faiss
# import numpy as np

# chunk_embeddings: shape (num_chunks, embedding_dim), already normalized to unit length
# dimension = chunk_embeddings.shape[1]
# index = faiss.IndexFlatIP(dimension)  # Inner Product ≈ cosine similarity when normalized
# index.add(chunk_embeddings)

### **2. Query phase**

In [ ]:
# query_embedding = embed_model.encode([question])
# query_embedding = normalize(query_embedding)  # match indexing normalization

# k = 5  # top-K chunks to retrieve
# distances, indices = index.search(query_embedding, k)
# retrieved_chunks = [chunk_list[i] for i in indices[0]]

### **Key details worth locking in:**

* ***index.add()*** — populates the index with all your chunk embeddings, done once during setup (or whenever new documents are added)
* ***index.search()*** — returns two arrays: distances (similarity/distance scores) and indices (positions of the matching chunks in your original chunk list) — you use indices to look up the actual text
* ***Normalization must be consistent*** — if you normalized chunk embeddings before building the index, you must also normalize the query embedding the same way, or your similarity scores become meaningless (comparing normalized vectors against un-normalized ones breaks the cosine-similarity equivalence from Day 27)
* ***Choosing K*** — too small risks missing the relevant chunk if it wasn't ranked #1; too large adds noise/irrelevant context to the reader/generator downstream. K=3–5 is a common practical range.

## **Combining Retriever with BERT-Style Reader**
### **The combined pipeline**

***Question → Retriever (FAISS) → Top-K relevant chunks →
Reader (BERT-style extractive QA) → Final extracted answer***

Instead of running extractive QA blindly on one chunk you already know contains the answer (Day 24's limitation), you now:

* Retrieve the top-K chunks most semantically similar to the question (Day 28's FAISS retriever)
* Read — run each retrieved chunk (or the concatenation of them) through a BERT-style extractive QA model, which predicts a start/end span as the answer

### **Two practical strategies for combining multiple retrieved chunks**

1. Strategy 1 — **Concatenate then read**:
Combine the top-K chunks into one longer context (up to the reader model's max sequence length), then run extractive QA once on the combined text.
    * Simple to implement
    * Risk: if K is large, you may exceed the reader's context limit, forcing truncation and potentially losing the chunk that actually had the answer

2. Strategy 2 — **Read each chunk separately**, pick the best answer:
Run the reader independently on each of the K retrieved chunks, get a candidate answer + confidence score from each, then return the answer with the highest confidence score across all chunks.

    * More robust — doesn't require squeezing everything into one context window
    * Slightly more compute (K separate reader passes instead of one)
    * This is generally the preferred approach in practice, and the one worth remembering for exam purposes

### **Why retrieval quality directly caps reader quality (an important exam-level insight)**

The reader can only be as good as the chunks it's given. If the retriever fails to surface the chunk that actually contains the answer (bad chunking, wrong K, embedding mismatch — all from Day 28's failure modes), the reader has no chance of finding the correct answer, no matter how good the reader model itself is. This is why RAG systems are usually debugged by first checking retrieval quality in isolation (does the top-K actually contain the answer-bearing chunk?) before ever suspecting the reader model.

### **Confidence scoring in extractive QA**

BERT-style QA models don't just output a span — they output probability scores for each possible start and end position. The overall "confidence" of a predicted answer span is typically derived from the combined probability of the chosen start and end positions. This confidence score is exactly what Strategy 2 uses to pick the best candidate answer across multiple chunks.

# **Advanced LangChain Components with Deep QA**

LangChain is a framework that wraps and orchestrates everything you've built so far (embeddings, retrievers, readers/generators) into reusable, chainable components — instead of writing all the glue code yourself.